In [ ]:
import scanpy as sc
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import pathlib as pl
import math
import tifffile

from matplotlib import patheffects

In [ ]:
from spatialfusion.embed.embed import AEInputs, run_full_embedding

# Helper functions Leiden

In [ ]:
from scipy.spatial.distance import cdist
import numpy as np


def merge_small_clusters_strict(
    adata,
    labels,
    min_size,
    use_rep="gcn",
):
    """
    Merge clusters smaller than min_size.
    Guarantees that all remaining clusters have size >= min_size.
    """
    labels = labels.astype(str).copy()

    while True:
        counts = labels.value_counts()
        small = counts[counts < min_size].index.tolist()

        if len(small) == 0:
            break

        # merge all small clusters into one
        labels[labels.isin(small)] = "__small__"

        counts = labels.value_counts()

        # if merged cluster is still too small → merge into nearest valid cluster
        if "__small__" in counts and counts["__small__"] < min_size:
            small_idx = labels == "__small__"
            small_centroid = adata.obsm[use_rep][small_idx].mean(axis=0)

            valid_clusters = [
                c for c in counts.index
                if c != "__small__"
            ]

            centroids = []
            for c in valid_clusters:
                idx = labels == c
                centroids.append(adata.obsm[use_rep][idx].mean(axis=0))

            dists = cdist([small_centroid], centroids)[0]
            nearest = valid_clusters[int(np.argmin(dists))]

            labels[labels == "__small__"] = nearest

        else:
            break

    return labels.astype("category")


def leiden_eval_exact_k(
    adata,
    resolution,
    min_size,
    use_rep="gcn",
    random_state=0,
):
    sc.tl.leiden(
        adata,
        resolution=float(resolution),
        flavor="igraph",
        n_iterations=2,
        random_state=random_state,
        key_added="_leiden_tmp",
    )

    merged = merge_small_clusters_strict(
        adata,
        adata.obs["_leiden_tmp"],
        min_size=min_size,
        use_rep=use_rep,
    )

    adata.obs.drop(columns="_leiden_tmp", inplace=True)

    return merged.nunique(), merged


def find_resolution_exact_k_binary(
    adata,
    k_target,
    min_size,
    use_rep="gcn",
    r_min=0.01,
    r_max=3.0,
    max_iter=12,
    tol=1e-3,
    random_state=0,
):
    sc.pp.neighbors(adata, use_rep=use_rep)

    lo, hi = r_min, r_max
    best_exact = None
    best_fallback = None  # (delta, resolution, labels)

    for i in range(max_iter):
        mid = (lo + hi) / 2
        ad = adata.copy()

        n_clusters, labels = leiden_eval_exact_k(
            ad,
            resolution=mid,
            min_size=min_size,
            use_rep=use_rep,
            random_state=random_state,
        )

        delta = n_clusters - k_target

        print(
            f"[iter {i}] res={mid:.4f} → "
            f"{n_clusters} clusters (Δ={delta})"
        )

        if n_clusters == k_target:
            best_exact = (mid, labels)
            break

        # store smallest overshoot
        if delta > 0:
            if best_fallback is None or delta < best_fallback[0]:
                best_fallback = (delta, mid, labels)
            hi = mid
        else:
            lo = mid

        if hi - lo < tol:
            break

    if best_exact is not None:
        return best_exact[0], best_exact[1]

    if best_fallback is not None:
        print(
            f"⚠️ No exact solution. Using fallback with Δ={best_fallback[0]}"
        )
        return best_fallback[1], best_fallback[2]

    raise RuntimeError("No valid clustering found.")


def merge_to_exact_k(
    adata,
    labels,
    k_target,
    use_rep="gcn",
):
    labels = labels.astype(str).copy()

    while labels.nunique() > k_target:
        counts = labels.value_counts()
        smallest = counts.idxmin()

        centroids = {}
        for c in labels.unique():
            idx = labels == c
            centroids[c] = adata.obsm[use_rep][idx].mean(axis=0)

        others = [c for c in centroids if c != smallest]
        dists = cdist(
            [centroids[smallest]],
            [centroids[c] for c in others],
        )[0]

        nearest = others[int(np.argmin(dists))]
        labels[labels == smallest] = nearest

    return labels.astype("category")


def leiden_exact_k_pipeline(
    adata,
    k_target,
    min_size,
    use_rep="gcn",
    random_state=0,
):
    res, labels = find_resolution_exact_k_binary(
        adata,
        k_target=k_target,
        min_size=min_size,
        use_rep=use_rep,
        random_state=random_state,
    )

    # if fallback was used, labels may have >k clusters
    if labels.nunique() > k_target:
        labels = merge_to_exact_k(
            adata,
            labels,
            k_target=k_target,
            use_rep=use_rep,
        )

    adata.obs["leiden"] = labels
    return adata, res


# Function to run 

In [ ]:
from tqdm.auto import tqdm
import time

In [ ]:
from pathlib import Path

def find_matching_ae_checkpoint(meta, ae_root):

    ae_root = Path(ae_root)

    he = meta["he_encoder"].lower()
    rna = meta["rna_encoder"].lower()
    align = meta["alignment_mode"].lower()

    matches = []

    for run_dir in ae_root.iterdir():

        if not run_dir.is_dir():
            continue

        name = run_dir.name.lower()

        if he not in name:
            continue

        if rna not in name:
            continue

        if align not in name:
            continue

        model_path = run_dir / "model.pt"

        if model_path.exists():
            matches.append(model_path)

    if len(matches) == 0:

        raise RuntimeError(
            f"No AE checkpoint found for "
            f"{he}/{rna}/{align}"
        )

    if len(matches) > 1:

        print(
            f"WARNING: found {len(matches)} matching AE checkpoints."
        )

        matches = sorted(
            matches,
            key=lambda p: p.parent.name
        )

    return matches[-1]

In [ ]:
import torch
import numpy as np

from spatialfusion.embed.embed import run_full_embedding
from spatialfusion.models.gcn import SpatialSmoothingBaseline
import yaml
import pandas as pd
import scanpy as sc
from pathlib import Path


# ============================================================
# CONFIG HELPERS
# ============================================================

def load_run_config(run_dir):

    run_dir = Path(run_dir)

    cfg_files = list(run_dir.glob("config_*.yaml"))

    if len(cfg_files) != 1:
        raise RuntimeError(
            f"Expected exactly one config_*.yaml in {run_dir}, "
            f"found {len(cfg_files)}"
        )

    return yaml.safe_load(cfg_files[0].read_text())

def parse_run_metadata(run_dir):

    cfg = load_run_config(run_dir)

    training = cfg["training"]
    eval_cfg = cfg["eval"]

    return {
        "run_dir": Path(run_dir),
        "model_path": Path(run_dir) / "model.pt",

        "model_type": training["model_type"],

        "he_encoder": training["he_encoder"],
        "rna_encoder": training["rna_encoder"],

        "alignment_mode": training["alignment_mode"],
        "combine_mode": training["combine_mode"],
        "pathway_mode": training["pathway_mode"],

        "hidden_dim": training["hidden_dim"],
        "num_layers": training["num_layers"],

        "embedding_dir": Path(eval_cfg["embedding_dir"]),
        "z1file": eval_cfg["z1file"],
        "z2file": eval_cfg["z2file"],

        "use_cls_loss": training["use_cls_loss"],
    }


def make_run_name(meta):

    cls_tag = (
        "cls"
        if meta["use_cls_loss"]
        else "nocls"
    )

    return (
        f"{meta['model_type']}_"
        f"{meta['he_encoder']}_"
        f"{meta['rna_encoder']}_"
        f"{meta['alignment_mode']}_"
        f"{meta['combine_mode']}_"
        f"{cls_tag}"
    )

from spatialfusion.embed.embed import AEInputs


def build_raw_inputs(
    sample_name,
    adata,
    meta,
    embedding_root,
):

    emb_dir = Path(embedding_root)

    he_encoder = meta["he_encoder"].lower()
    rna_encoder = meta["rna_encoder"].lower()

    he_map = {
        "uni": "UNI.parquet",
        "virchow": "Virchow2.parquet",
    }

    rna_map = {
        "scgpt": "scGPT.parquet",
        "nicheformer": "nicheformer.parquet",
    }

    he_file = emb_dir / he_map[he_encoder]
    rna_file = emb_dir / rna_map[rna_encoder]

    print(f"Loading H&E embeddings: {he_file}")
    print(f"Loading RNA embeddings: {rna_file}")

    z_he = pd.read_parquet(he_file)
    z_rna = pd.read_parquet(rna_file)

    return {
        sample_name: AEInputs(
            adata=adata,
            z_uni=z_he,
            z_scgpt=z_rna,
        )
    }

from spatialfusion.embed.embed import run_full_embedding
from spatialfusion.models.gcn import SpatialSmoothingBaseline
import torch


def compute_graph_embedding(

    adata,
    ae_inputs,

    ae_model_path,
    graph_model_path,

    model_type,

    combine_mode,
    hidden_dim,
    num_layers,

    device="cuda:0",
    spatial_key="spatial_px",
):

    gcn_model = None
    gcn_model_path = None

    if model_type == "gcn":

        gcn_model_path = graph_model_path

    elif model_type == "smoothing":

        model = SpatialSmoothingBaseline(
            node_mask_ratio=0.0
        )

        try:

            state = torch.load(
                graph_model_path,
                map_location=device,
            )

            model.load_state_dict(
                state,
                strict=False,
            )

        except Exception:
            pass

        model.eval()
        model.to(device)

        gcn_model = model

    else:

        raise ValueError(model_type)

    embeddings_df = run_full_embedding(

        ae_inputs_by_sample=ae_inputs,

        ae_model_path=ae_model_path,

        gcn_model=gcn_model,
        gcn_model_path=gcn_model_path,

        combine_mode=combine_mode,
        hidden_dim=hidden_dim,
        num_layers=num_layers,

        spatial_key=spatial_key,
        device=device,

        save_ae_dir=None,
    )

    embeddings_df = embeddings_df.set_index("cell_id")

    metadata_cols = [
        "sample_id",
        "cell_id",
        "celltype",
        "cellsubtypes",
        "CNiche",
        "TNiche",
        "X_coord",
        "Y_coord",
    ]

    embeddings_df = embeddings_df.drop(
        columns=[
            c
            for c in metadata_cols
            if c in embeddings_df.columns
        ]
    )

    adata.obsm["gcn"] = embeddings_df.loc[
        adata.obs_names
    ].values.astype(np.float32)

    return adata

# ============================================================
# MODEL EXECUTION
# ============================================================


def run_single_model(

    adata,
    sample_name,

    meta,

    ae_root,
    embedding_root,

    device,
    spatial_key,
):

    ae_ckpt = find_matching_ae_checkpoint(
        meta,
        ae_root,
    )

    print("\n" + "=" * 100)
    print(make_run_name(meta))
    print("=" * 100)

    print(f"AE model     : {ae_ckpt}")
    print(f"Graph model  : {meta['model_path']}")

    raw_inputs = build_raw_inputs(
        sample_name=sample_name,
        adata=adata,
        meta=meta,
        embedding_root=embedding_root,
    )

    ad = adata.copy()

    ad = compute_graph_embedding(

        adata=ad,

        ae_inputs=raw_inputs,

        ae_model_path=ae_ckpt,
        graph_model_path=meta["model_path"],

        model_type=meta["model_type"],
        combine_mode=meta["combine_mode"],
        hidden_dim=meta["hidden_dim"],
        num_layers=meta["num_layers"],

        device=device,
        spatial_key=spatial_key,
    )

    return ad


# ============================================================
# MAIN LOOP
# ============================================================

def run_all_models(

    adata,
    sample_name,

    gcn_root,
    ae_root,
    embedding_root,

    out_dir,

    min_cluster_size=500,
    k_target=None,

    device="cuda:0",
    spatial_key="spatial_px",
):

    run_dirs = sorted(
        d
        for d in Path(gcn_root).iterdir()
        if d.is_dir()
    )

    print(f"\nFound {len(run_dirs)} graph runs\n")

    for run_dir in tqdm(
        run_dirs,
        desc="Models",
    ):

        try:

            meta = parse_run_metadata(run_dir)

            run_name = make_run_name(meta)
            
            emb_path = (
                pl.Path(out_dir) /
                f"{run_name}_embeddings.parquet"
            )
            
            cluster_path = (
                pl.Path(out_dir) /
                f"{run_name}_clusters.csv"
            )
            
            # Skip if already completed
            if emb_path.exists() and cluster_path.exists():
                print(f"✓ Already completed: {run_name}")
                continue

            ad = run_single_model(

                adata=adata,
                sample_name=sample_name,

                meta=meta,

                ae_root=ae_root,
                embedding_root=embedding_root,

                device=device,
                spatial_key=spatial_key,
            )

            ad, resolution = leiden_exact_k_pipeline(
                ad,
                min_size=min_cluster_size,
                k_target=k_target,
            )

            run_name = make_run_name(meta)

            emb_file = (
                Path(out_dir)
                / f"{run_name}_embeddings.parquet"
            )

            cluster_file = (
                Path(out_dir)
                / f"{run_name}_clusters.csv"
            )

            pd.DataFrame(
                ad.obsm["gcn"],
                index=ad.obs_names,
            ).to_parquet(
                emb_file
            )

            ad.obs[
                ["leiden"]
            ].to_csv(
                cluster_file
            )

            print(
                f"\n✓ Saved {run_name}"
            )

        except Exception as e:

            print(
                f"\nFAILED: {run_dir.name}"
            )

            print(e)

In [ ]:
sample_name = 'TENXOv5k'

In [ ]:
rawdata_ovca = sc.read_h5ad('../../..Broad_SpatialFoundation/test_data/10X_Xenium_Ovarian_5k/adata.h5ad')

region_annot = pd.read_csv('../../..Broad_SpatialFoundation/test_data/10X_Xenium_Ovarian_5k/region_annotations.csv',index_col=0)

rawdata_ovca.obs['path_region'] = region_annot.loc[rawdata_ovca.obs_names].values.ravel()

rawdata_ovca.obs = pd.concat([rawdata_ovca.obs, pd.DataFrame(rawdata_ovca.obsm['spatial_px'], index=rawdata_ovca.obs_names, columns=['X_coord','Y_coord'])],axis=1)

region_df = rawdata_ovca.obs[['cell_labels', 'minor_celltype', 'major_celltype', 'cell_id',
       'path_region', 'X_coord','Y_coord']]

region_df['sample_id'] = 'TENXOv5k'

pathway_matrix = pd.read_parquet('../../..Broad_SpatialFoundation/test_data/10X_Xenium_Ovarian_5k/pathway_activation.parquet')

In [ ]:
adata = rawdata_ovca.copy()
adata.obs["sample_id"] = sample_name

## k=11

In [ ]:
## RUNNING GCN 

run_all_models(
    adata=adata,

    sample_name=sample_name,

    ae_root="../../..SpatialFusion/results/ae_encoder_sweep/",
    gcn_root="../../..SpatialFusion/results/gcn_fusion_sweep/",
    embedding_root="../../..Broad_SpatialFoundation/test_data/10X_Xenium_Ovarian_5k/embeddings/",

    out_dir=(
        "../../..Broad_SpatialFoundation/test_data/"
        "10X_Xenium_Ovarian_5k/"
        "embeddings_hyperparameter_search"
    ),

    min_cluster_size=500,
    k_target=11,

    device="cuda:4",

    spatial_key="spatial_px",
)

In [ ]:
## RUNNING SMOOTHING

run_all_models(
    adata=adata,

    sample_name=sample_name,

    ae_root="../../..SpatialFusion/results/ae_encoder_sweep/",
    gcn_root="../../..SpatialFusion/results/smoothing_fusion_sweep/",
    embedding_root="../../..Broad_SpatialFoundation/test_data/10X_Xenium_Ovarian_5k/embeddings/",

    out_dir=(
        "../../..Broad_SpatialFoundation/test_data/"
        "10X_Xenium_Ovarian_5k/"
        "embeddings_hyperparameter_search"
    ),

    min_cluster_size=500,
    k_target=11,

    device="cuda:4",

    spatial_key="spatial_px",
)

## For k=13

In [ ]:
## RUNNING GCN 

run_all_models(
    adata=adata,

    sample_name=sample_name,

    ae_root="../../..SpatialFusion/results/ae_encoder_sweep/",
    gcn_root="../../..SpatialFusion/results/gcn_fusion_sweep/",
    embedding_root="../../..Broad_SpatialFoundation/test_data/10X_Xenium_Ovarian_5k/embeddings/",

    out_dir=(
        "../../..Broad_SpatialFoundation/test_data/"
        "10X_Xenium_Ovarian_5k/"
        "embeddings_hyperparameter_search_k_13"
    ),

    min_cluster_size=500,
    k_target=13,

    device="cuda:4",

    spatial_key="spatial_px",
)

In [ ]:
## RUNNING SMOOTHING

run_all_models(
    adata=adata,

    sample_name=sample_name,

    ae_root="../../..SpatialFusion/results/ae_encoder_sweep/",
    gcn_root="../../..SpatialFusion/results/smoothing_fusion_sweep/",
    embedding_root="../../..Broad_SpatialFoundation/test_data/10X_Xenium_Ovarian_5k/embeddings/",

    out_dir=(
        "../../..Broad_SpatialFoundation/test_data/"
        "10X_Xenium_Ovarian_5k/"
         "embeddings_hyperparameter_search_k_13"
    ),

    min_cluster_size=500,
    k_target=13,

    device="cuda:4",

    spatial_key="spatial_px",
)

## For k=9

In [ ]:
## RUNNING GCN 

run_all_models(
    adata=adata,

    sample_name=sample_name,

    ae_root="../../..SpatialFusion/results/ae_encoder_sweep/",
    gcn_root="../../..SpatialFusion/results/gcn_fusion_sweep/",
    embedding_root="../../..Broad_SpatialFoundation/test_data/10X_Xenium_Ovarian_5k/embeddings/",

    out_dir=(
        "../../..Broad_SpatialFoundation/test_data/"
        "10X_Xenium_Ovarian_5k/"
        "embeddings_hyperparameter_search_k_9"
    ),

    min_cluster_size=500,
    k_target=9,

    device="cuda:4",

    spatial_key="spatial_px",
)

In [ ]:
## RUNNING SMOOTHING

run_all_models(
    adata=adata,

    sample_name=sample_name,

    ae_root="../../..SpatialFusion/results/ae_encoder_sweep/",
    gcn_root="../../..SpatialFusion/results/smoothing_fusion_sweep/",
    embedding_root="../../..Broad_SpatialFoundation/test_data/10X_Xenium_Ovarian_5k/embeddings/",

    out_dir=(
        "../../..Broad_SpatialFoundation/test_data/"
        "10X_Xenium_Ovarian_5k/"
         "embeddings_hyperparameter_search_k_9"
    ),

    min_cluster_size=500,
    k_target=9,

    device="cuda:4",

    spatial_key="spatial_px",
)

# Compute scores 

In [ ]:
import numpy as np
import pandas as pd
from tqdm import tqdm

import scanpy as sc

from scipy.spatial import distance
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import *


In [ ]:
def compute_PAS_fast(clusterlabel, location, k=10):
    clusterlabel = np.array(clusterlabel)
    location = np.array(location)

    # Fit NearestNeighbors (ignore self-match later)
    nbrs = NearestNeighbors(n_neighbors=k+1, algorithm='auto').fit(location)
    distances, indices = nbrs.kneighbors(location)

    # Remove self (first column is self in most cases)
    neighbor_indices = indices[:, 1:]  # shape: (n_samples, k)

    # Check PAS condition
    mismatches = np.array([
        np.sum(clusterlabel[neighbor_indices[i]] != clusterlabel[i]) > (k / 2)
        for i in range(len(clusterlabel))
    ])

    return np.sum(mismatches) / len(clusterlabel)


def compute_CHAOS_fast(clusterlabel, location):

    clusterlabel = np.asarray(clusterlabel).astype(str)
    
    location = np.array(location)
    matched_location = StandardScaler().fit_transform(location)

    clusterlabel_unique = np.unique(clusterlabel)
    dist_val = 0
    total_count = 0

    for k in tqdm(clusterlabel_unique, desc="Computing CHAOS"):
        cluster_mask = clusterlabel == k
        location_cluster = matched_location[cluster_mask]
        n = location_cluster.shape[0]

        if n <= 2:
            continue

        # Use NearestNeighbors to find 1-NN distances
        nbrs = NearestNeighbors(n_neighbors=2, algorithm='auto').fit(location_cluster)
        distances, _ = nbrs.kneighbors(location_cluster)

        # distances[:, 0] is zero (self), distances[:, 1] is nearest neighbor
        dist_val += np.sum(distances[:, 1])
        total_count += n

    return dist_val / total_count if total_count > 0 else np.nan


def compute_ASW_fast(adata, pred_key, spatial_key='spatial'):
    coords = adata.obsm[spatial_key]
    labels = adata.obs[pred_key]
    return silhouette_score(X=coords, labels=labels, metric='euclidean')

def compute_ARI(adata,gt_key,pred_key):
    y_true = adata.obs[gt_key].astype(str)
    y_pred = adata.obs[pred_key].astype(str)

    return adjusted_rand_score(y_true,y_pred)

def compute_NMI(adata,gt_key,pred_key):
    y_true = adata.obs[gt_key].astype(str)
    y_pred = adata.obs[pred_key].astype(str)
    
    return normalized_mutual_info_score(y_true,y_pred)

def compute_HOM(adata,gt_key,pred_key):
    y_true = adata.obs[gt_key].astype(str)
    y_pred = adata.obs[pred_key].astype(str)
    
    return homogeneity_score(y_true,y_pred)

def compute_COM(adata,gt_key,pred_key):

    y_true = adata.obs[gt_key].astype(str)
    y_pred = adata.obs[pred_key].astype(str)
    
    return completeness_score(y_true,y_pred)

def compute_all_metrics(adata, clustering_keys, ground_truth_key='path_region', spatial_key='spatial_px'):
    results = {}

    for method_name, cluster_key in clustering_keys.items():
        metrics = {
            'ARI': compute_ARI(adata, cluster_key, ground_truth_key),
            'NMI': compute_NMI(adata, cluster_key, ground_truth_key),
            'HOM': compute_HOM(adata, cluster_key, ground_truth_key),
            'COM': compute_COM(adata, cluster_key, ground_truth_key),
            'PAS': compute_PAS_fast(adata.obs[cluster_key], adata.obsm[spatial_key]),
            'CHAOS': compute_CHAOS_fast(adata.obs[cluster_key], adata.obsm[spatial_key]),
        }
        results[method_name] = metrics

    return pd.DataFrame(results)

import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
from matplotlib.colors import LinearSegmentedColormap

def format_number(value):
    """Format numbers: scientific notation if <0.01, else 2 decimals."""
    if pd.isna(value):
        return ""
    if abs(value) < 0.01 and value != 0:
        return f"{value:.0e}"  # 1 decimal in scientific notation, e.g. 3.4e-04
    else:
        return f"{value:.2f}"  # two decimals otherwise

def plot_benchmark_heatmap(
    results_df,
    title="Spatial clustering benchmark",
    savefig=None,
    metric_order=None,
    figsize=None,
):
    """
    Nature Genetics–style benchmarking heatmap showing method rankings across metrics.
    Allows manual control of metric order.
    """

    lower_better = {'PAS', 'CHAOS'}

    # --- Default metric order ---
    if metric_order is None:
        metric_order = list(results_df.index)

    # --- Normalize scores ---
    df_norm = results_df.copy()
    for metric in df_norm.index:
        vals = df_norm.loc[metric]
        if metric in lower_better:
            vals = -vals
        df_norm.loc[metric] = (vals - vals.min()) / (vals.max() - vals.min() + 1e-9)

    # --- Rank per metric ---
    ranks = results_df.copy()
    for metric in ranks.index:
        ranks.loc[metric] = results_df.loc[metric].rank(ascending=(metric in lower_better))

    # --- Prepare longform for plotting ---
    df_plot = df_norm.reset_index().melt(
        id_vars='index', var_name='Method', value_name='Normalized'
    ).rename(columns={'index': 'Metric'})

    df_plot['Raw'] = results_df.reset_index().melt(
        id_vars='index', var_name='Method', value_name='Raw'
    )['Raw']

    df_plot['Rank'] = ranks.reset_index().melt(
        id_vars='index', var_name='Method', value_name='Rank'
    )['Rank']

    # Add directional arrows
    df_plot['MetricLabel'] = df_plot['Metric'].apply(
        lambda m: f"{m} {'↓' if m in lower_better else '↑'}"
    )

    # --- Construct ordered MetricLabel list ---
    metric_order_labels = []
    for m in metric_order:
        arrow = '↓' if m in lower_better else '↑'
        metric_order_labels.append(f"{m} {arrow}")

    # --- Heatmap data matrix ---
    method_order = results_df.columns.tolist()
    df_matrix = df_plot.pivot_table(
        index="MetricLabel", columns="Method", values="Normalized"
    ).loc[metric_order_labels, method_order]

    # --- Aesthetics ---
    sns.set_theme(style="white", context="talk")
    if figsize is None:
        figsize=(1.3 * len(method_order), 0.8 * len(metric_order))
                 
    fig, ax = plt.subplots(figsize=figsize, dpi=300)
    # Enhance contrast near the top (gamma correction)
    gamma = 3  ### THIS IS ONLY FOR THE COLOR FOR PLOTTING PURPOSES, NOT THE NUMBERS!
    df_matrix_contrast = df_matrix ** gamma
    sns.heatmap(
        df_matrix_contrast,
        #cmap="vlag",
        cmap = LinearSegmentedColormap.from_list(
            "vlag_red",
            ["#fee8ef",  # very light pink
             "#f4a3a8",  # pastel red
             "#d95858",  # mid red
             "#b40426"]  # vlag red (vivid crimson)
        ),
        cbar=False,
        ax=ax,
        linewidths=0,
        square=True,
    )

    # --- Adaptive text color (white on dark, black on light) ---
    #cmap = plt.get_cmap("vlag")
    cmap = LinearSegmentedColormap.from_list(
        "vlag_red",
        ["#fee8ef",  # very light pink
         "#f4a3a8",  # pastel red
         "#d95858",  # mid red
         "#b40426"]  # vlag red (vivid crimson)
    )

    for i, metric in enumerate(df_matrix.index):
        base_metric = metric.split()[0]
        for j, method in enumerate(df_matrix.columns):
            raw_val = results_df.loc[base_metric, method]
            norm_val = df_matrix.loc[metric, method]

            # Compute luminance for adaptive color
            rgb = np.array(cmap(norm_val)[:3])
            luminance = 0.2126 * rgb[0] + 0.7152 * rgb[1] + 0.0722 * rgb[2]
            text_color = "black" if luminance > 0.5 else "white"

            ax.text(
                j + 0.5, i + 0.5,
                format_number(raw_val),
                ha='center', va='center',
                color=text_color,
                fontsize=8,
                fontweight='normal',
            )

    # --- Formatting ---
    ax.set_title(title, fontsize=10, pad=14, fontweight='normal')
    ax.set_xlabel("")
    ax.set_ylabel("")
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right", fontsize=10, fontweight='normal')
    ax.set_yticklabels(ax.get_yticklabels(), fontsize=10, fontweight='normal')

    for spine in ax.spines.values():
        spine.set_visible(False)

    plt.tight_layout()

    if savefig:
        fig.savefig(
            savefig,
            bbox_inches="tight",
            dpi=300,
            format=savefig.split('.')[-1],
            transparent=True
        )
        print(f"Saved: {savefig}")

    plt.show()



# K=13

In [ ]:
adata= rawdata_ovca.copy()

outdir = "../../..Broad_SpatialFoundation/test_data/10X_Xenium_Ovarian_5k/embeddings_hyperparameter_search_k_13/"
adata_obs = adata.obs.copy()
cols = []
for f in pl.Path(outdir).iterdir():
    if '.ipynb' in str(f):
        continue
    elif '_clusters' in str(f):
        name = f.stem.split('_')[:-1]
        name = ' '.join(name)
        cols.append(name)
        clustering = pd.read_csv(pl.Path(outdir) / f'{f.stem}.csv', index_col=0)
        clustering.columns = [name]
        adata_obs = pd.concat([adata_obs, clustering],axis=1)

In [ ]:
adata.obs = adata_obs

In [ ]:
model_type_order = {
    "gcn": 0,
    "smoothing": 1,
}

input_order = {
    ("uni", "scgpt"): 0,
    ("uni", "nicheformer"): 1,
    ("virchow", "scgpt"): 2,
    ("virchow", "nicheformer"): 3,
}

recon_order = {
    "full": 0,
    "recon only": 1,
}

fusion_order = {
    "gated": 0,
    "average": 1,
    "concat": 2,
}

cls_order = {
    "cls": 0,
    "nocls": 1,
}


def sort_key(name):

    parts = name.split()

    # gcn uni scgpt full average cls
    #  0   1    2      3      4     5

    he_encoder = parts[1]
    rna_encoder = parts[2]

    if parts[3] == "recon":
        recon_mode = "recon only"
        fusion_mode = parts[5]
        cls_mode = parts[6]
    else:
        recon_mode = "full"
        fusion_mode = parts[4]
        cls_mode = parts[5]

    return (
        model_type_order[parts[0]],
        input_order[(he_encoder, rna_encoder)],
        recon_order[recon_mode],
        fusion_order[fusion_mode],
        cls_order[cls_mode],
    )

In [ ]:
sorted_cols = sorted(cols)


In [ ]:
clustering_keys = {
    v: v for v in sorted_cols
}

results_df = compute_all_metrics(adata, clustering_keys)

In [ ]:
results_df.to_csv('results_hyperparameter_k13.csv')

# K=11

In [ ]:
adata= rawdata_ovca.copy()

outdir = "../../..Broad_SpatialFoundation/test_data/10X_Xenium_Ovarian_5k/embeddings_hyperparameter_search/"
adata_obs = adata.obs.copy()
cols = []
for f in pl.Path(outdir).iterdir():
    if '.ipynb' in str(f):
        continue
    elif '_clusters' in str(f):
        name = f.stem.split('_')[:-1]
        name = ' '.join(name)
        cols.append(name)
        clustering = pd.read_csv(pl.Path(outdir) / f'{f.stem}.csv', index_col=0)
        clustering.columns = [name]
        adata_obs = pd.concat([adata_obs, clustering],axis=1)

In [ ]:
adata.obs = adata_obs

In [ ]:
model_type_order = {
    "gcn": 0,
    "smoothing": 1,
}

input_order = {
    ("uni", "scgpt"): 0,
    ("uni", "nicheformer"): 1,
    ("virchow", "scgpt"): 2,
    ("virchow", "nicheformer"): 3,
}

recon_order = {
    "full": 0,
    "recon only": 1,
}

fusion_order = {
    "gated": 0,
    "average": 1,
    "concat": 2,
}

cls_order = {
    "cls": 0,
    "nocls": 1,
}


def sort_key(name):

    parts = name.split()

    # gcn uni scgpt full average cls
    #  0   1    2      3      4     5

    he_encoder = parts[1]
    rna_encoder = parts[2]

    if parts[3] == "recon":
        recon_mode = "recon only"
        fusion_mode = parts[5]
        cls_mode = parts[6]
    else:
        recon_mode = "full"
        fusion_mode = parts[4]
        cls_mode = parts[5]

    return (
        model_type_order[parts[0]],
        input_order[(he_encoder, rna_encoder)],
        recon_order[recon_mode],
        fusion_order[fusion_mode],
        cls_order[cls_mode],
    )

In [ ]:
sorted_cols = sorted(cols)


In [ ]:
clustering_keys = {
    v: v for v in sorted_cols
}

results_df = compute_all_metrics(adata, clustering_keys)

In [ ]:
results_df.to_csv('results_hyperparameter.csv')

# K=9

In [ ]:
adata= rawdata_ovca.copy()

outdir = "../../..Broad_SpatialFoundation/test_data/10X_Xenium_Ovarian_5k/embeddings_hyperparameter_search_k_9/"
adata_obs = adata.obs.copy()
cols = []
for f in pl.Path(outdir).iterdir():
    if '.ipynb' in str(f):
        continue
    elif '_clusters' in str(f):
        name = f.stem.split('_')[:-1]
        name = ' '.join(name)
        cols.append(name)
        clustering = pd.read_csv(pl.Path(outdir) / f'{f.stem}.csv', index_col=0)
        clustering.columns = [name]
        adata_obs = pd.concat([adata_obs, clustering],axis=1)

In [ ]:
adata.obs = adata_obs

In [ ]:
model_type_order = {
    "gcn": 0,
    "smoothing": 1,
}

input_order = {
    ("uni", "scgpt"): 0,
    ("uni", "nicheformer"): 1,
    ("virchow", "scgpt"): 2,
    ("virchow", "nicheformer"): 3,
}

recon_order = {
    "full": 0,
    "recon only": 1,
}

fusion_order = {
    "gated": 0,
    "average": 1,
    "concat": 2,
}

cls_order = {
    "cls": 0,
    "nocls": 1,
}


def sort_key(name):

    parts = name.split()

    # gcn uni scgpt full average cls
    #  0   1    2      3      4     5

    he_encoder = parts[1]
    rna_encoder = parts[2]

    if parts[3] == "recon":
        recon_mode = "recon only"
        fusion_mode = parts[5]
        cls_mode = parts[6]
    else:
        recon_mode = "full"
        fusion_mode = parts[4]
        cls_mode = parts[5]

    return (
        model_type_order[parts[0]],
        input_order[(he_encoder, rna_encoder)],
        recon_order[recon_mode],
        fusion_order[fusion_mode],
        cls_order[cls_mode],
    )

In [ ]:
sorted_cols = sorted(cols)


In [ ]:
clustering_keys = {
    v: v for v in sorted_cols
}

results_df = compute_all_metrics(adata, clustering_keys)

In [ ]:
results_df.to_csv('results_hyperparameter_k9.csv')

# Now compare

In [ ]:
import pandas as pd

def parse_model_name(name):

    parts = name.split()

    graph = parts[0]
    he = parts[1]
    rna = parts[2]

    if parts[3] == "recon":
        recon = "recon"
        fusion = parts[5]
        cls = parts[6]
    else:
        recon = "full"
        fusion = parts[4]
        cls = parts[5]

    return {
        "model": name,
        "graph": graph,
        "he": he,
        "rna": rna,
        "recon": recon,
        "fusion": fusion,
        "cls": cls,
    }

def compute_overall_score(
    results_df,
    exclude_metrics=None,
):
    """
    Compute normalized overall score across metrics.

    Parameters
    ----------
    results_df : pd.DataFrame
        rows = metrics
        cols = models

    exclude_metrics : list or None
        Metrics to exclude from score calculation.
        Example:
            ["PAS", "CHAOS"]

    Returns
    -------
    overall_score : pd.Series
        One score per model.
    """

    if exclude_metrics is None:
        exclude_metrics = []

    lower_better = {"PAS", "CHAOS"}

    score_df = results_df.copy()

    # remove unwanted metrics
    score_df = score_df.drop(
        index=[
            m for m in exclude_metrics
            if m in score_df.index
        ]
    )

    for metric in score_df.index:

        vals = score_df.loc[metric]

        if metric in lower_better:
            vals = -vals

        score_df.loc[metric] = (
            vals - vals.min()
        ) / (
            vals.max() - vals.min() + 1e-12
        )

    overall_score = score_df.mean(axis=0)

    return overall_score

import numpy as np
import pandas as pd


def paired_effect_table(meta, score_col="bio_score"):

    results = []

    comparisons = [

        {
            "factor": "cls",
            "positive": "cls",
            "negative": "nocls",
            "groupby": [
                "graph",
                "he",
                "rna",
                "recon",
                "fusion",
            ],
        },

        {
            "factor": "recon",
            "positive": "full",
            "negative": "recon",
            "groupby": [
                "graph",
                "he",
                "rna",
                "fusion",
                "cls",
            ],
        },

        {
            "factor": "fusion_concat_vs_average",
            "positive": "concat",
            "negative": "average",
            "column": "fusion",
            "groupby": [
                "graph",
                "he",
                "rna",
                "recon",
                "cls",
            ],
        },

        {
            "factor": "fusion_gated_vs_average",
            "positive": "gated",
            "negative": "average",
            "column": "fusion",
            "groupby": [
                "graph",
                "he",
                "rna",
                "recon",
                "cls",
            ],
        },

        {
            "factor": "fusion_gated_vs_concat",
            "positive": "gated",
            "negative": "concat",
            "column": "fusion",
            "groupby": [
                "graph",
                "he",
                "rna",
                "recon",
                "cls",
            ],
        },

        {
            "factor": "he",
            "positive": "virchow",
            "negative": "uni",
            "groupby": [
                "graph",
                "rna",
                "recon",
                "fusion",
                "cls",
            ],
        },

        {
            "factor": "rna",
            "positive": "scgpt",
            "negative": "nicheformer",
            "groupby": [
                "graph",
                "he",
                "recon",
                "fusion",
                "cls",
            ],
        },

        {
            "factor": "graph",
            "positive": "gcn",
            "negative": "smoothing",
            "groupby": [
                "he",
                "rna",
                "recon",
                "fusion",
                "cls",
            ],
        },
    ]

    for comp in comparisons:

        deltas = []

        factor = comp["factor"]

        if factor.startswith("fusion"):

            column = "fusion"

        elif factor == "cls":

            column = "cls"

        elif factor == "recon":

            column = "recon"

        elif factor == "he":

            column = "he"

        elif factor == "rna":

            column = "rna"

        elif factor == "graph":

            column = "graph"

        for _, group in meta.groupby(comp["groupby"]):

            pos = group[group[column] == comp["positive"]]
            neg = group[group[column] == comp["negative"]]

            if len(pos) != 1:
                continue

            if len(neg) != 1:
                continue

            delta = (
                pos[score_col].iloc[0]
                - neg[score_col].iloc[0]
            )

            deltas.append(delta)

        deltas = np.asarray(deltas)

        results.append({
            "comparison":
                f"{comp['positive']} - {comp['negative']}",

            "n_pairs":
                len(deltas),

            "mean_delta":
                np.mean(deltas),

            "median_delta":
                np.median(deltas),

            "wins":
                np.sum(deltas > 0),

            "win_rate":
                np.mean(deltas > 0),
        })

    return (
        pd.DataFrame(results)
        .sort_values(
            "mean_delta",
            ascending=False
        )
    )

# k=11

In [ ]:
results_df = pd.read_csv('results_hyperparameter.csv', index_col=0)

meta = pd.DataFrame(
    [parse_model_name(c) for c in results_df.columns]
)

meta.head()

overall_score = compute_overall_score(results_df)

meta["score"] = meta["model"].map(overall_score)

bio_score = compute_overall_score(
    results_df,
    exclude_metrics=["PAS", "CHAOS"]
)

meta["bio_score"] = meta["model"].map(bio_score)

for metric in ['ARI','NMI','COM','HOM']:
    meta[metric] = meta["model"].map(results_df.loc[metric])

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

metrics = ["bio_score", "ARI", "NMI", "COM", "HOM"]

sns.set_theme(
    style="white",
    context="paper",
    font_scale=1.3
)

fig, axes = plt.subplots(
    1,
    len(metrics),
    figsize=(18, 4),
    constrained_layout=True
)

for ax, metric in zip(axes, metrics):

    plot_df = meta

    sns.boxplot(
        data=plot_df,
        x="graph",
        y=metric,
        order=["gcn","smoothing"],
        width=0.6,
        showcaps=True,
        showfliers=False,
        boxprops={"facecolor": "white"},
        ax=ax,
    )

    sns.stripplot(
        data=plot_df,
        x="graph",
        y=metric,
        order=["gcn","smoothing"],
        color="black",
        alpha=0.5,
        size=4,
        jitter=0.15,
        ax=ax,
    )

    ax.set_title(metric, fontsize=14, fontweight="bold")
    ax.set_xlabel("")
    ax.set_ylabel(metric)

    sns.despine(ax=ax)

fig.supxlabel("Model type", fontsize=14)
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

metrics = ["bio_score", "ARI", "NMI", "COM", "HOM"]

sns.set_theme(
    style="white",
    context="paper",
    font_scale=1.3
)

fig, axes = plt.subplots(
    1,
    len(metrics),
    figsize=(18, 4),
    constrained_layout=True
)

for ax, metric in zip(axes, metrics):

    plot_df = meta[meta.graph == "gcn"]

    sns.boxplot(
        data=plot_df,
        x="recon",
        y=metric,
        order=["full", "recon"],
        width=0.6,
        showcaps=True,
        showfliers=False,
        boxprops={"facecolor": "white"},
        ax=ax,
    )

    sns.stripplot(
        data=plot_df,
        x="recon",
        y=metric,
        order=["full", "recon"],
        color="black",
        alpha=0.5,
        size=4,
        jitter=0.15,
        ax=ax,
    )

    ax.set_title(metric, fontsize=14, fontweight="bold")
    ax.set_xlabel("")
    ax.set_ylabel(metric)

    sns.despine(ax=ax)

fig.supxlabel("Alignment objective", fontsize=14)
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

metrics = ["bio_score", "ARI", "NMI", "COM", "HOM"]

sns.set_theme(
    style="white",
    context="paper",
    font_scale=1.3
)

fig, axes = plt.subplots(
    1,
    len(metrics),
    figsize=(18, 4),
    constrained_layout=True
)

for ax, metric in zip(axes, metrics):

    plot_df = meta[meta.graph == "gcn"]

    sns.boxplot(
        data=plot_df,
        x="fusion",
        y=metric,
        order=["average","concat","gated"],
        width=0.6,
        showcaps=True,
        showfliers=False,
        boxprops={"facecolor": "white"},
        ax=ax,
    )

    sns.stripplot(
        data=plot_df,
        x="fusion",
        y=metric,
        order=["average","concat","gated"],
        color="black",
        alpha=0.5,
        size=4,
        jitter=0.15,
        ax=ax,
    )

    ax.set_title(metric, fontsize=14, fontweight="bold")
    ax.set_xlabel("")
    ax.set_ylabel(metric)

    sns.despine(ax=ax)

fig.supxlabel("Fusion type", fontsize=14)
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

metrics = ["bio_score", "ARI", "NMI", "COM", "HOM"]

sns.set_theme(
    style="white",
    context="paper",
    font_scale=1.3
)

fig, axes = plt.subplots(
    1,
    len(metrics),
    figsize=(18, 4),
    constrained_layout=True
)

for ax, metric in zip(axes, metrics):

    plot_df = meta[meta.graph == "gcn"]

    sns.boxplot(
        data=plot_df,
        x="he",
        y=metric,
        order=["uni","virchow"],
        width=0.6,
        showcaps=True,
        showfliers=False,
        boxprops={"facecolor": "white"},
        ax=ax,
    )

    sns.stripplot(
        data=plot_df,
        x="he",
        y=metric,
        order=["uni","virchow"],
        color="black",
        alpha=0.5,
        size=4,
        jitter=0.15,
        ax=ax,
    )

    ax.set_title(metric, fontsize=14, fontweight="bold")
    ax.set_xlabel("")
    ax.set_ylabel(metric)

    sns.despine(ax=ax)

fig.supxlabel("H&E FM input", fontsize=14)
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

metrics = ["bio_score", "ARI", "NMI", "COM", "HOM"]

sns.set_theme(
    style="white",
    context="paper",
    font_scale=1.3
)

fig, axes = plt.subplots(
    1,
    len(metrics),
    figsize=(18, 4),
    constrained_layout=True
)

for ax, metric in zip(axes, metrics):

    plot_df = meta[meta.graph == "gcn"]

    sns.boxplot(
        data=plot_df,
        x="rna",
        y=metric,
        order=["scgpt","nicheformer"],
        width=0.6,
        showcaps=True,
        showfliers=False,
        boxprops={"facecolor": "white"},
        ax=ax,
    )

    sns.stripplot(
        data=plot_df,
        x="rna",
        y=metric,
        order=["scgpt","nicheformer"],
        color="black",
        alpha=0.5,
        size=4,
        jitter=0.15,
        ax=ax,
    )

    ax.set_title(metric, fontsize=14, fontweight="bold")
    ax.set_xlabel("")
    ax.set_ylabel(metric)

    sns.despine(ax=ax)

fig.supxlabel("RNA FM input", fontsize=14)
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

metrics = ["bio_score", "ARI", "NMI", "COM", "HOM"]

sns.set_theme(
    style="white",
    context="paper",
    font_scale=1.3
)

fig, axes = plt.subplots(
    1,
    len(metrics),
    figsize=(18, 4),
    constrained_layout=True
)

for ax, metric in zip(axes, metrics):

    plot_df = meta[meta.graph == "gcn"]

    sns.boxplot(
        data=plot_df,
        x="cls",
        y=metric,
        order=["cls","nocls"],
        width=0.6,
        showcaps=True,
        showfliers=False,
        boxprops={"facecolor": "white"},
        ax=ax,
    )

    sns.stripplot(
        data=plot_df,
        x="cls",
        y=metric,
        order=["cls","nocls"],
        color="black",
        alpha=0.5,
        size=4,
        jitter=0.15,
        ax=ax,
    )

    ax.set_title(metric, fontsize=14, fontweight="bold")
    ax.set_xlabel("")
    ax.set_ylabel(metric)

    sns.despine(ax=ax)

fig.supxlabel("Auxiliary regression loss", fontsize=14)
plt.show()

In [ ]:
effect_df = paired_effect_table(
    meta,
    score_col="bio_score",
)

effect_df

In [ ]:
effect_df = paired_effect_table(
    meta[meta.graph=='gcn'],
    score_col="bio_score",
)

effect_df

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plot_df = (
    effect_df
    .dropna()
    .sort_values("mean_delta", ascending=True)
    .copy()
)

plt.figure(figsize=(5, 3))

sns.barplot(
    data=plot_df,
    x="mean_delta",
    y="comparison",
)

plt.axvline(
    0,
    color="black",
    linestyle="--",
    linewidth=1,
)

plt.xlabel("Mean paired Δ bio_score")
plt.ylabel("")
plt.title("Effect of architectural choices")

for i, row in enumerate(plot_df.itertuples()):
    sign = 1 if row.mean_delta>0 else -1
    plt.text(
        row.mean_delta + sign * 0.001,
        i,
        f"  {row.mean_delta:.2f}",
        va="center",
    )

plt.tight_layout()

plt.savefig('pairwise_comparison_bioscore_k11.svg', dpi=300)
plt.show()

In [ ]:
effect_df = paired_effect_table(
    meta[meta.graph=='gcn'],
    score_col="ARI",
)

effect_df

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plot_df = (
    effect_df
    .dropna()
    .sort_values("mean_delta", ascending=True)
    .copy()
)

plt.figure(figsize=(5, 3))

sns.barplot(
    data=plot_df,
    x="mean_delta",
    y="comparison",
)

plt.axvline(
    0,
    color="black",
    linestyle="--",
    linewidth=1,
)

plt.xlabel("Mean paired Δ ARI")
plt.ylabel("")
plt.title("Effect of architectural choices")

for i, row in enumerate(plot_df.itertuples()):
    sign = 1 if row.mean_delta>0 else -1
    plt.text(
        row.mean_delta + sign * 0.001,
        i,
        f"  {row.mean_delta:.2f}",
        va="center",
    )

plt.tight_layout()

plt.savefig('pairwise_comparison_ari_k11.svg', dpi=300)
plt.show()

# k=9

In [ ]:
results_df = pd.read_csv('results_hyperparameter_k9.csv', index_col=0)

meta = pd.DataFrame(
    [parse_model_name(c) for c in results_df.columns]
)

meta.head()

overall_score = compute_overall_score(results_df)

meta["score"] = meta["model"].map(overall_score)

bio_score = compute_overall_score(
    results_df,
    exclude_metrics=["PAS", "CHAOS"]
)

meta["bio_score"] = meta["model"].map(bio_score)

for metric in ['ARI','NMI','COM','HOM']:
    meta[metric] = meta["model"].map(results_df.loc[metric])

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

metrics = ["bio_score", "ARI", "NMI", "COM", "HOM"]

sns.set_theme(
    style="white",
    context="paper",
    font_scale=1.3
)

fig, axes = plt.subplots(
    1,
    len(metrics),
    figsize=(18, 4),
    constrained_layout=True
)

for ax, metric in zip(axes, metrics):

    plot_df = meta

    sns.boxplot(
        data=plot_df,
        x="graph",
        y=metric,
        order=["gcn","smoothing"],
        width=0.6,
        showcaps=True,
        showfliers=False,
        boxprops={"facecolor": "white"},
        ax=ax,
    )

    sns.stripplot(
        data=plot_df,
        x="graph",
        y=metric,
        order=["gcn","smoothing"],
        color="black",
        alpha=0.5,
        size=4,
        jitter=0.15,
        ax=ax,
    )

    ax.set_title(metric, fontsize=14, fontweight="bold")
    ax.set_xlabel("")
    ax.set_ylabel(metric)

    sns.despine(ax=ax)

fig.supxlabel("Model type", fontsize=14)
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

metrics = ["bio_score", "ARI", "NMI", "COM", "HOM"]

sns.set_theme(
    style="white",
    context="paper",
    font_scale=1.3
)

fig, axes = plt.subplots(
    1,
    len(metrics),
    figsize=(18, 4),
    constrained_layout=True
)

for ax, metric in zip(axes, metrics):

    plot_df = meta[meta.graph == "gcn"]

    sns.boxplot(
        data=plot_df,
        x="recon",
        y=metric,
        order=["full", "recon"],
        width=0.6,
        showcaps=True,
        showfliers=False,
        boxprops={"facecolor": "white"},
        ax=ax,
    )

    sns.stripplot(
        data=plot_df,
        x="recon",
        y=metric,
        order=["full", "recon"],
        color="black",
        alpha=0.5,
        size=4,
        jitter=0.15,
        ax=ax,
    )

    ax.set_title(metric, fontsize=14, fontweight="bold")
    ax.set_xlabel("")
    ax.set_ylabel(metric)

    sns.despine(ax=ax)

fig.supxlabel("Alignment objective", fontsize=14)
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

metrics = ["bio_score", "ARI", "NMI", "COM", "HOM"]

sns.set_theme(
    style="white",
    context="paper",
    font_scale=1.3
)

fig, axes = plt.subplots(
    1,
    len(metrics),
    figsize=(18, 4),
    constrained_layout=True
)

for ax, metric in zip(axes, metrics):

    plot_df = meta[meta.graph == "gcn"]

    sns.boxplot(
        data=plot_df,
        x="fusion",
        y=metric,
        order=["average","concat","gated"],
        width=0.6,
        showcaps=True,
        showfliers=False,
        boxprops={"facecolor": "white"},
        ax=ax,
    )

    sns.stripplot(
        data=plot_df,
        x="fusion",
        y=metric,
        order=["average","concat","gated"],
        color="black",
        alpha=0.5,
        size=4,
        jitter=0.15,
        ax=ax,
    )

    ax.set_title(metric, fontsize=14, fontweight="bold")
    ax.set_xlabel("")
    ax.set_ylabel(metric)

    sns.despine(ax=ax)

fig.supxlabel("Fusion type", fontsize=14)
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

metrics = ["bio_score", "ARI", "NMI", "COM", "HOM"]

sns.set_theme(
    style="white",
    context="paper",
    font_scale=1.3
)

fig, axes = plt.subplots(
    1,
    len(metrics),
    figsize=(18, 4),
    constrained_layout=True
)

for ax, metric in zip(axes, metrics):

    plot_df = meta[meta.graph == "gcn"]

    sns.boxplot(
        data=plot_df,
        x="he",
        y=metric,
        order=["uni","virchow"],
        width=0.6,
        showcaps=True,
        showfliers=False,
        boxprops={"facecolor": "white"},
        ax=ax,
    )

    sns.stripplot(
        data=plot_df,
        x="he",
        y=metric,
        order=["uni","virchow"],
        color="black",
        alpha=0.5,
        size=4,
        jitter=0.15,
        ax=ax,
    )

    ax.set_title(metric, fontsize=14, fontweight="bold")
    ax.set_xlabel("")
    ax.set_ylabel(metric)

    sns.despine(ax=ax)

fig.supxlabel("H&E FM input", fontsize=14)
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

metrics = ["bio_score", "ARI", "NMI", "COM", "HOM"]

sns.set_theme(
    style="white",
    context="paper",
    font_scale=1.3
)

fig, axes = plt.subplots(
    1,
    len(metrics),
    figsize=(18, 4),
    constrained_layout=True
)

for ax, metric in zip(axes, metrics):

    plot_df = meta[meta.graph == "gcn"]

    sns.boxplot(
        data=plot_df,
        x="rna",
        y=metric,
        order=["scgpt","nicheformer"],
        width=0.6,
        showcaps=True,
        showfliers=False,
        boxprops={"facecolor": "white"},
        ax=ax,
    )

    sns.stripplot(
        data=plot_df,
        x="rna",
        y=metric,
        order=["scgpt","nicheformer"],
        color="black",
        alpha=0.5,
        size=4,
        jitter=0.15,
        ax=ax,
    )

    ax.set_title(metric, fontsize=14, fontweight="bold")
    ax.set_xlabel("")
    ax.set_ylabel(metric)

    sns.despine(ax=ax)

fig.supxlabel("RNA FM input", fontsize=14)
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

metrics = ["bio_score", "ARI", "NMI", "COM", "HOM"]

sns.set_theme(
    style="white",
    context="paper",
    font_scale=1.3
)

fig, axes = plt.subplots(
    1,
    len(metrics),
    figsize=(18, 4),
    constrained_layout=True
)

for ax, metric in zip(axes, metrics):

    plot_df = meta[meta.graph == "gcn"]

    sns.boxplot(
        data=plot_df,
        x="cls",
        y=metric,
        order=["cls","nocls"],
        width=0.6,
        showcaps=True,
        showfliers=False,
        boxprops={"facecolor": "white"},
        ax=ax,
    )

    sns.stripplot(
        data=plot_df,
        x="cls",
        y=metric,
        order=["cls","nocls"],
        color="black",
        alpha=0.5,
        size=4,
        jitter=0.15,
        ax=ax,
    )

    ax.set_title(metric, fontsize=14, fontweight="bold")
    ax.set_xlabel("")
    ax.set_ylabel(metric)

    sns.despine(ax=ax)

fig.supxlabel("Auxiliary regression loss", fontsize=14)
plt.show()

In [ ]:
effect_df = paired_effect_table(
    meta,
    score_col="bio_score",
)

effect_df

In [ ]:
effect_df = paired_effect_table(
    meta[meta.graph=='gcn'],
    score_col="bio_score",
)

effect_df

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plot_df = (
    effect_df
    .dropna()
    .sort_values("mean_delta", ascending=True)
    .copy()
)

plt.figure(figsize=(5, 3))

sns.barplot(
    data=plot_df,
    x="mean_delta",
    y="comparison",
)

plt.axvline(
    0,
    color="black",
    linestyle="--",
    linewidth=1,
)

plt.xlabel("Mean paired Δ bio_score")
plt.ylabel("")
plt.title("Effect of architectural choices")

for i, row in enumerate(plot_df.itertuples()):
    sign = 1 if row.mean_delta>0 else -1
    plt.text(
        row.mean_delta + sign * 0.001,
        i,
        f"  {row.mean_delta:.2f}",
        va="center",
    )

plt.tight_layout()

plt.savefig('pairwise_comparison_bioscore_k9.svg', dpi=300)
plt.show()

In [ ]:
effect_df = paired_effect_table(
    meta[meta.graph=='gcn'],
    score_col="ARI",
)

effect_df

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plot_df = (
    effect_df
    .dropna()
    .sort_values("mean_delta", ascending=True)
    .copy()
)

plt.figure(figsize=(5, 3))

sns.barplot(
    data=plot_df,
    x="mean_delta",
    y="comparison",
)

plt.axvline(
    0,
    color="black",
    linestyle="--",
    linewidth=1,
)

plt.xlabel("Mean paired Δ ARI")
plt.ylabel("")
plt.title("Effect of architectural choices")

for i, row in enumerate(plot_df.itertuples()):
    sign = 1 if row.mean_delta>0 else -1
    plt.text(
        row.mean_delta + sign * 0.001,
        i,
        f"  {row.mean_delta:.2f}",
        va="center",
    )

plt.tight_layout()

plt.savefig('pairwise_comparison_ari_k9.svg', dpi=300)
plt.show()

# k=13

In [ ]:
results_df = pd.read_csv('results_hyperparameter_k13.csv', index_col=0)

meta = pd.DataFrame(
    [parse_model_name(c) for c in results_df.columns]
)

meta.head()

overall_score = compute_overall_score(results_df)

meta["score"] = meta["model"].map(overall_score)

bio_score = compute_overall_score(
    results_df,
    exclude_metrics=["PAS", "CHAOS"]
)

meta["bio_score"] = meta["model"].map(bio_score)

for metric in ['ARI','NMI','COM','HOM']:
    meta[metric] = meta["model"].map(results_df.loc[metric])

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

metrics = ["bio_score", "ARI", "NMI", "COM", "HOM"]

sns.set_theme(
    style="white",
    context="paper",
    font_scale=1.3
)

fig, axes = plt.subplots(
    1,
    len(metrics),
    figsize=(18, 4),
    constrained_layout=True
)

for ax, metric in zip(axes, metrics):

    plot_df = meta

    sns.boxplot(
        data=plot_df,
        x="graph",
        y=metric,
        order=["gcn","smoothing"],
        width=0.6,
        showcaps=True,
        showfliers=False,
        boxprops={"facecolor": "white"},
        ax=ax,
    )

    sns.stripplot(
        data=plot_df,
        x="graph",
        y=metric,
        order=["gcn","smoothing"],
        color="black",
        alpha=0.5,
        size=4,
        jitter=0.15,
        ax=ax,
    )

    ax.set_title(metric, fontsize=14, fontweight="bold")
    ax.set_xlabel("")
    ax.set_ylabel(metric)

    sns.despine(ax=ax)

fig.supxlabel("Model type", fontsize=14)
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

metrics = ["bio_score", "ARI", "NMI", "COM", "HOM"]

sns.set_theme(
    style="white",
    context="paper",
    font_scale=1.3
)

fig, axes = plt.subplots(
    1,
    len(metrics),
    figsize=(18, 4),
    constrained_layout=True
)

for ax, metric in zip(axes, metrics):

    plot_df = meta[meta.graph == "gcn"]

    sns.boxplot(
        data=plot_df,
        x="recon",
        y=metric,
        order=["full", "recon"],
        width=0.6,
        showcaps=True,
        showfliers=False,
        boxprops={"facecolor": "white"},
        ax=ax,
    )

    sns.stripplot(
        data=plot_df,
        x="recon",
        y=metric,
        order=["full", "recon"],
        color="black",
        alpha=0.5,
        size=4,
        jitter=0.15,
        ax=ax,
    )

    ax.set_title(metric, fontsize=14, fontweight="bold")
    ax.set_xlabel("")
    ax.set_ylabel(metric)

    sns.despine(ax=ax)

fig.supxlabel("Alignment objective", fontsize=14)
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

metrics = ["bio_score", "ARI", "NMI", "COM", "HOM"]

sns.set_theme(
    style="white",
    context="paper",
    font_scale=1.3
)

fig, axes = plt.subplots(
    1,
    len(metrics),
    figsize=(18, 4),
    constrained_layout=True
)

for ax, metric in zip(axes, metrics):

    plot_df = meta[meta.graph == "gcn"]

    sns.boxplot(
        data=plot_df,
        x="fusion",
        y=metric,
        order=["average","concat","gated"],
        width=0.6,
        showcaps=True,
        showfliers=False,
        boxprops={"facecolor": "white"},
        ax=ax,
    )

    sns.stripplot(
        data=plot_df,
        x="fusion",
        y=metric,
        order=["average","concat","gated"],
        color="black",
        alpha=0.5,
        size=4,
        jitter=0.15,
        ax=ax,
    )

    ax.set_title(metric, fontsize=14, fontweight="bold")
    ax.set_xlabel("")
    ax.set_ylabel(metric)

    sns.despine(ax=ax)

fig.supxlabel("Fusion type", fontsize=14)
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

metrics = ["bio_score", "ARI", "NMI", "COM", "HOM"]

sns.set_theme(
    style="white",
    context="paper",
    font_scale=1.3
)

fig, axes = plt.subplots(
    1,
    len(metrics),
    figsize=(18, 4),
    constrained_layout=True
)

for ax, metric in zip(axes, metrics):

    plot_df = meta[meta.graph == "gcn"]

    sns.boxplot(
        data=plot_df,
        x="he",
        y=metric,
        order=["uni","virchow"],
        width=0.6,
        showcaps=True,
        showfliers=False,
        boxprops={"facecolor": "white"},
        ax=ax,
    )

    sns.stripplot(
        data=plot_df,
        x="he",
        y=metric,
        order=["uni","virchow"],
        color="black",
        alpha=0.5,
        size=4,
        jitter=0.15,
        ax=ax,
    )

    ax.set_title(metric, fontsize=14, fontweight="bold")
    ax.set_xlabel("")
    ax.set_ylabel(metric)

    sns.despine(ax=ax)

fig.supxlabel("H&E FM input", fontsize=14)
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

metrics = ["bio_score", "ARI", "NMI", "COM", "HOM"]

sns.set_theme(
    style="white",
    context="paper",
    font_scale=1.3
)

fig, axes = plt.subplots(
    1,
    len(metrics),
    figsize=(18, 4),
    constrained_layout=True
)

for ax, metric in zip(axes, metrics):

    plot_df = meta[meta.graph == "gcn"]

    sns.boxplot(
        data=plot_df,
        x="rna",
        y=metric,
        order=["scgpt","nicheformer"],
        width=0.6,
        showcaps=True,
        showfliers=False,
        boxprops={"facecolor": "white"},
        ax=ax,
    )

    sns.stripplot(
        data=plot_df,
        x="rna",
        y=metric,
        order=["scgpt","nicheformer"],
        color="black",
        alpha=0.5,
        size=4,
        jitter=0.15,
        ax=ax,
    )

    ax.set_title(metric, fontsize=14, fontweight="bold")
    ax.set_xlabel("")
    ax.set_ylabel(metric)

    sns.despine(ax=ax)

fig.supxlabel("RNA FM input", fontsize=14)
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

metrics = ["bio_score", "ARI", "NMI", "COM", "HOM"]

sns.set_theme(
    style="white",
    context="paper",
    font_scale=1.3
)

fig, axes = plt.subplots(
    1,
    len(metrics),
    figsize=(18, 4),
    constrained_layout=True
)

for ax, metric in zip(axes, metrics):

    plot_df = meta[meta.graph == "gcn"]

    sns.boxplot(
        data=plot_df,
        x="cls",
        y=metric,
        order=["cls","nocls"],
        width=0.6,
        showcaps=True,
        showfliers=False,
        boxprops={"facecolor": "white"},
        ax=ax,
    )

    sns.stripplot(
        data=plot_df,
        x="cls",
        y=metric,
        order=["cls","nocls"],
        color="black",
        alpha=0.5,
        size=4,
        jitter=0.15,
        ax=ax,
    )

    ax.set_title(metric, fontsize=14, fontweight="bold")
    ax.set_xlabel("")
    ax.set_ylabel(metric)

    sns.despine(ax=ax)

fig.supxlabel("Auxiliary regression loss", fontsize=14)
plt.show()

In [ ]:
effect_df = paired_effect_table(
    meta,
    score_col="bio_score",
)

effect_df

In [ ]:
effect_df = paired_effect_table(
    meta[meta.graph=='gcn'],
    score_col="bio_score",
)

effect_df

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plot_df = (
    effect_df
    .dropna()
    .sort_values("mean_delta", ascending=True)
    .copy()
)

plt.figure(figsize=(5, 3))

sns.barplot(
    data=plot_df,
    x="mean_delta",
    y="comparison",
)

plt.axvline(
    0,
    color="black",
    linestyle="--",
    linewidth=1,
)

plt.xlabel("Mean paired Δ bio_score")
plt.ylabel("")
plt.title("Effect of architectural choices")

for i, row in enumerate(plot_df.itertuples()):
    sign = 1 if row.mean_delta>0 else -1
    plt.text(
        row.mean_delta + sign * 0.001,
        i,
        f"  {row.mean_delta:.2f}",
        va="center",
    )

plt.tight_layout()

plt.savefig('pairwise_comparison_bioscore_k13.svg', dpi=300)
plt.show()

In [ ]:
effect_df = paired_effect_table(
    meta[meta.graph=='gcn'],
    score_col="ARI",
)

effect_df

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plot_df = (
    effect_df
    .dropna()
    .sort_values("mean_delta", ascending=True)
    .copy()
)

plt.figure(figsize=(5, 3))

sns.barplot(
    data=plot_df,
    x="mean_delta",
    y="comparison",
)

plt.axvline(
    0,
    color="black",
    linestyle="--",
    linewidth=1,
)

plt.xlabel("Mean paired Δ ARI")
plt.ylabel("")
plt.title("Effect of architectural choices")

for i, row in enumerate(plot_df.itertuples()):
    sign = 1 if row.mean_delta>0 else -1
    plt.text(
        row.mean_delta + sign * 0.001,
        i,
        f"  {row.mean_delta:.2f}",
        va="center",
    )

plt.tight_layout()

plt.savefig('pairwise_comparison_ari_k13.svg', dpi=300)
plt.show()

In [ ]:
effect_df = paired_effect_table(
    meta[meta.graph=='gcn'],
    score_col="NMI",
)

effect_df

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plot_df = (
    effect_df
    .dropna()
    .sort_values("mean_delta", ascending=True)
    .copy()
)

plt.figure(figsize=(5, 3))

sns.barplot(
    data=plot_df,
    x="mean_delta",
    y="comparison",
)

plt.axvline(
    0,
    color="black",
    linestyle="--",
    linewidth=1,
)

plt.xlabel("Mean paired Δ NMI")
plt.ylabel("")
plt.title("Effect of architectural choices")

for i, row in enumerate(plot_df.itertuples()):
    sign = 1 if row.mean_delta>0 else -1
    plt.text(
        row.mean_delta + sign * 0.001,
        i,
        f"  {row.mean_delta:.2f}",
        va="center",
    )

plt.tight_layout()

plt.savefig('pairwise_comparison_nmi_k13.svg', dpi=300)
plt.show()

In [ ]:
effect_df = paired_effect_table(
    meta[meta.graph=='gcn'],
    score_col="COM",
)

effect_df

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plot_df = (
    effect_df
    .dropna()
    .sort_values("mean_delta", ascending=True)
    .copy()
)

plt.figure(figsize=(5, 3))

sns.barplot(
    data=plot_df,
    x="mean_delta",
    y="comparison",
)

plt.axvline(
    0,
    color="black",
    linestyle="--",
    linewidth=1,
)

plt.xlabel("Mean paired Δ COM")
plt.ylabel("")
plt.title("Effect of architectural choices")

for i, row in enumerate(plot_df.itertuples()):
    sign = 1 if row.mean_delta>0 else -1
    plt.text(
        row.mean_delta + sign * 0.001,
        i,
        f"  {row.mean_delta:.2f}",
        va="center",
    )

plt.tight_layout()

plt.savefig('pairwise_comparison_com_k13.svg', dpi=300)
plt.show()

In [ ]:
effect_df = paired_effect_table(
    meta[meta.graph=='gcn'],
    score_col="HOM",
)

effect_df

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plot_df = (
    effect_df
    .dropna()
    .sort_values("mean_delta", ascending=True)
    .copy()
)

plt.figure(figsize=(5, 3))

sns.barplot(
    data=plot_df,
    x="mean_delta",
    y="comparison",
)

plt.axvline(
    0,
    color="black",
    linestyle="--",
    linewidth=1,
)

plt.xlabel("Mean paired Δ HOM")
plt.ylabel("")
plt.title("Effect of architectural choices")

for i, row in enumerate(plot_df.itertuples()):
    sign = 1 if row.mean_delta>0 else -1
    plt.text(
        row.mean_delta + sign * 0.001,
        i,
        f"  {row.mean_delta:.2f}",
        va="center",
    )

plt.tight_layout()

plt.savefig('pairwise_comparison_hom_k13.svg', dpi=300)
plt.show()